#02_silver_ops_quality.sql

Operação / Observabilidade - Camada Silver

Este notebook consolida a saúde da camada silver e suportar operação em ambiente corporativo:
- volumes (silver/rejects)
- taxa de rejeição total e do último load
- top motivos de rejeição (último load)
- checkpoints (última execução por tabela)
- histórico de snapchots (para tendência / alertas)

##Objetivo
- Monitorar execução e qualidade por tabela
- Acelerar troubleshooting (top reject_reason + volumes)
- Criar base para alertas e SLOs de dados

##Estratégia
- SQL-only
- último load definido por max (ingestion_ts) por tabela
- snapshots gravados em silver_ops.quality_report_history

##Contexto

In [0]:
USE CATALOG healthcare_dev;

##Cria schemas

In [0]:
CREATE SCHEMA IF NOT EXISTS healthcare_dev.silver_ops;

##Armazena relatório por execução

In [0]:
CREATE TABLE IF NOT EXISTS healthcare_dev.silver_ops.quality_report_history (
  report_ts TIMESTAMP,
  table_name STRING,

  -- totais
  silver_count BIGINT,
  rejects_count BIGINT,
  rejects_pct DOUBLE,

  -- último load
  last_load_id STRING,
  silver_count_last_load BIGINT,
  rejects_count_last_load BIGINT,
  rejects_pct_last_load DOUBLE,

  -- top reject do último load
  top_reject_reason_last_load STRING,
  top_reject_count_last_load BIGINT
)
USING DELTA;


##Tabela de volumes silver e rejects

In [0]:
CREATE OR REPLACE TEMP VIEW v_table_catalog AS
SELECT 'dim_beneficiarios' AS table_name,
       'healthcare_dev.silver.dim_beneficiarios' AS silver_fqn,
       'healthcare_dev.silver_rejects.dim_beneficiarios' AS rejects_fqn
UNION ALL
SELECT 'fact_contratos',
       'healthcare_dev.silver.fact_contratos',
       'healthcare_dev.silver_rejects.fact_contratos'
UNION ALL
SELECT 'fact_faturas',
       'healthcare_dev.silver.fact_faturas',
       'healthcare_dev.silver_rejects.fact_faturas'
UNION ALL
SELECT 'fact_app_logs',
       'healthcare_dev.silver.fact_app_logs',
       'healthcare_dev.silver_rejects.fact_app_logs'
UNION ALL
SELECT 'fact_sac',
       'healthcare_dev.silver.fact_sac',
       'healthcare_dev.silver_rejects.fact_sac'
UNION ALL
SELECT 'fact_eventos',
       'healthcare_dev.silver.fact_eventos',
       'healthcare_dev.silver_rejects.fact_eventos';

##Métricas totais (silver/rejects) por tabela

In [0]:
CREATE OR REPLACE TEMP VIEW v_quality_totals AS
SELECT
  'dim_beneficiarios' AS table_name,
  (SELECT COUNT(*) FROM healthcare_dev.silver.dim_beneficiarios) AS silver_count,
  (SELECT COUNT(*) FROM healthcare_dev.silver_rejects.dim_beneficiarios) AS rejects_count
UNION ALL
SELECT
  'fact_contratos',
  (SELECT COUNT(*) FROM healthcare_dev.silver.fact_contratos),
  (SELECT COUNT(*) FROM healthcare_dev.silver_rejects.fact_contratos)
UNION ALL
SELECT
  'fact_faturas',
  (SELECT COUNT(*) FROM healthcare_dev.silver.fact_faturas),
  (SELECT COUNT(*) FROM healthcare_dev.silver_rejects.fact_faturas)
UNION ALL
SELECT
  'fact_app_logs',
  (SELECT COUNT(*) FROM healthcare_dev.silver.fact_app_logs),
  (SELECT COUNT(*) FROM healthcare_dev.silver_rejects.fact_app_logs)
UNION ALL
SELECT
  'fact_sac',
  (SELECT COUNT(*) FROM healthcare_dev.silver.fact_sac),
  (SELECT COUNT(*) FROM healthcare_dev.silver_rejects.fact_sac)
UNION ALL
SELECT
  'fact_eventos',
  (SELECT COUNT(*) FROM healthcare_dev.silver.fact_eventos),
  (SELECT COUNT(*) FROM healthcare_dev.silver_rejects.fact_eventos);

In [0]:
CREATE OR REPLACE TEMP VIEW v_quality_totals_enriched AS
SELECT
  table_name,
  silver_count,
  rejects_count,
  ROUND(100.0 * rejects_count / NULLIF(silver_count + rejects_count, 0), 4) AS rejects_pct
FROM v_quality_totals;

##último load_id por tabela (por maior ingestion_ts na silver)

In [0]:
CREATE OR REPLACE TEMP VIEW v_last_load_ids AS
SELECT 'dim_beneficiarios' AS table_name,
       (SELECT load_id
        FROM healthcare_dev.silver.dim_beneficiarios
        WHERE load_id IS NOT NULL
        ORDER BY ingestion_ts DESC
        LIMIT 1) AS last_load_id
UNION ALL
SELECT 'fact_contratos',
       (SELECT load_id
        FROM healthcare_dev.silver.fact_contratos
        WHERE load_id IS NOT NULL
        ORDER BY ingestion_ts DESC
        LIMIT 1)
UNION ALL
SELECT 'fact_faturas',
       (SELECT load_id
        FROM healthcare_dev.silver.fact_faturas
        WHERE load_id IS NOT NULL
        ORDER BY ingestion_ts DESC
        LIMIT 1)
UNION ALL
SELECT 'fact_app_logs',
       (SELECT load_id
        FROM healthcare_dev.silver.fact_app_logs
        WHERE load_id IS NOT NULL
        ORDER BY ingestion_ts DESC
        LIMIT 1)
UNION ALL
SELECT 'fact_sac',
       (SELECT load_id
        FROM healthcare_dev.silver.fact_sac
        WHERE load_id IS NOT NULL
        ORDER BY ingestion_ts DESC
        LIMIT 1)
UNION ALL
SELECT 'fact_eventos',
       (SELECT load_id
        FROM healthcare_dev.silver.fact_eventos
        WHERE load_id IS NOT NULL
        ORDER BY ingestion_ts DESC
        LIMIT 1);

##Métricas do último load (silver/rejects) por tabela

In [0]:
CREATE OR REPLACE TEMP VIEW v_quality_last_load AS
SELECT
  l.table_name,
  l.last_load_id,

  -- silver last load
  CASE l.table_name
    WHEN 'dim_beneficiarios' THEN (SELECT COUNT(*) FROM healthcare_dev.silver.dim_beneficiarios WHERE load_id = l.last_load_id)
    WHEN 'fact_contratos'    THEN (SELECT COUNT(*) FROM healthcare_dev.silver.fact_contratos    WHERE load_id = l.last_load_id)
    WHEN 'fact_faturas'      THEN (SELECT COUNT(*) FROM healthcare_dev.silver.fact_faturas      WHERE load_id = l.last_load_id)
    WHEN 'fact_app_logs'     THEN (SELECT COUNT(*) FROM healthcare_dev.silver.fact_app_logs     WHERE load_id = l.last_load_id)
    WHEN 'fact_sac'          THEN (SELECT COUNT(*) FROM healthcare_dev.silver.fact_sac          WHERE load_id = l.last_load_id)
    WHEN 'fact_eventos'      THEN (SELECT COUNT(*) FROM healthcare_dev.silver.fact_eventos      WHERE load_id = l.last_load_id)
  END AS silver_count_last_load,
  -- rejects last load
  CASE l.table_name
    WHEN 'dim_beneficiarios' THEN (SELECT COUNT(*) FROM healthcare_dev.silver_rejects.dim_beneficiarios WHERE load_id = l.last_load_id)
    WHEN 'fact_contratos'    THEN (SELECT COUNT(*) FROM healthcare_dev.silver_rejects.fact_contratos    WHERE load_id = l.last_load_id)
    WHEN 'fact_faturas'      THEN (SELECT COUNT(*) FROM healthcare_dev.silver_rejects.fact_faturas      WHERE load_id = l.last_load_id)
    WHEN 'fact_app_logs'     THEN (SELECT COUNT(*) FROM healthcare_dev.silver_rejects.fact_app_logs     WHERE load_id = l.last_load_id)
    WHEN 'fact_sac'          THEN (SELECT COUNT(*) FROM healthcare_dev.silver_rejects.fact_sac          WHERE load_id = l.last_load_id)
    WHEN 'fact_eventos'      THEN (SELECT COUNT(*) FROM healthcare_dev.silver_rejects.fact_eventos      WHERE load_id = l.last_load_id)
  END AS rejects_count_last_load

FROM v_last_load_ids l;


In [0]:
CREATE OR REPLACE TEMP VIEW v_quality_last_load_enriched AS
SELECT
  table_name,
  last_load_id,
  silver_count_last_load,
  rejects_count_last_load,
  ROUND(100.0 * rejects_count_last_load / NULLIF(silver_count_last_load + rejects_count_last_load, 0), 4) AS rejects_pct_last_load
FROM v_quality_last_load;

##Top reject_reason do último load (por tabela)

In [0]:
CREATE OR REPLACE TEMP VIEW v_top_reject_last_load AS
SELECT
  'fact_contratos' AS table_name,
  (SELECT reject_reason FROM healthcare_dev.silver_rejects.fact_contratos r
    JOIN (SELECT last_load_id FROM v_quality_last_load_enriched WHERE table_name='fact_contratos') x
      ON r.load_id = x.last_load_id
   GROUP BY reject_reason
   ORDER BY COUNT(*) DESC
   LIMIT 1) AS top_reject_reason_last_load,
  (SELECT COUNT(*) FROM healthcare_dev.silver_rejects.fact_contratos r
    JOIN (SELECT last_load_id FROM v_quality_last_load_enriched WHERE table_name='fact_contratos') x
      ON r.load_id = x.last_load_id
   GROUP BY reject_reason
   ORDER BY COUNT(*) DESC
   LIMIT 1) AS top_reject_count_last_load
UNION ALL
SELECT
  'fact_faturas',
  (SELECT reject_reason FROM healthcare_dev.silver_rejects.fact_faturas r
    JOIN (SELECT last_load_id FROM v_quality_last_load_enriched WHERE table_name='fact_faturas') x
      ON r.load_id = x.last_load_id
   GROUP BY reject_reason
   ORDER BY COUNT(*) DESC
   LIMIT 1),
  (SELECT COUNT(*) FROM healthcare_dev.silver_rejects.fact_faturas r
    JOIN (SELECT last_load_id FROM v_quality_last_load_enriched WHERE table_name='fact_faturas') x
      ON r.load_id = x.last_load_id
   GROUP BY reject_reason
   ORDER BY COUNT(*) DESC
   LIMIT 1)
  
UNION ALL
SELECT
  'fact_app_logs',
  (SELECT reject_reason FROM healthcare_dev.silver_rejects.fact_app_logs r
    JOIN (SELECT last_load_id FROM v_quality_last_load_enriched WHERE table_name='fact_app_logs') x
      ON r.load_id = x.last_load_id
   GROUP BY reject_reason
   ORDER BY COUNT(*) DESC
   LIMIT 1),
  (SELECT COUNT(*) FROM healthcare_dev.silver_rejects.fact_app_logs r
    JOIN (SELECT last_load_id FROM v_quality_last_load_enriched WHERE table_name='fact_app_logs') x
      ON r.load_id = x.last_load_id
   GROUP BY reject_reason
   ORDER BY COUNT(*) DESC
   LIMIT 1)
UNION ALL
SELECT
  'fact_sac',
  (SELECT reject_reason FROM healthcare_dev.silver_rejects.fact_sac r
    JOIN (SELECT last_load_id FROM v_quality_last_load_enriched WHERE table_name='fact_sac') x
      ON r.load_id = x.last_load_id
   GROUP BY reject_reason
   ORDER BY COUNT(*) DESC
   LIMIT 1),
  (SELECT COUNT(*) FROM healthcare_dev.silver_rejects.fact_sac r
    JOIN (SELECT last_load_id FROM v_quality_last_load_enriched WHERE table_name='fact_sac') x
      ON r.load_id = x.last_load_id
   GROUP BY reject_reason
   ORDER BY COUNT(*) DESC
   LIMIT 1)
UNION ALL
SELECT
  'fact_eventos',
  (SELECT reject_reason FROM healthcare_dev.silver_rejects.fact_eventos r
    JOIN (SELECT last_load_id FROM v_quality_last_load_enriched WHERE table_name='fact_eventos') x
      ON r.load_id = x.last_load_id
   GROUP BY reject_reason
   ORDER BY COUNT(*) DESC
   LIMIT 1),
  (SELECT COUNT(*) FROM healthcare_dev.silver_rejects.fact_eventos r
    JOIN (SELECT last_load_id FROM v_quality_last_load_enriched WHERE table_name='fact_eventos') x
      ON r.load_id = x.last_load_id
   GROUP BY reject_reason
   ORDER BY COUNT(*) DESC
   LIMIT 1)
UNION ALL
SELECT
  'dim_beneficiarios',
  (SELECT reject_reason FROM healthcare_dev.silver_rejects.dim_beneficiarios r
    JOIN (SELECT last_load_id FROM v_quality_last_load_enriched WHERE table_name='dim_beneficiarios') x
      ON r.load_id = x.last_load_id
   GROUP BY reject_reason
   ORDER BY COUNT(*) DESC
   LIMIT 1),
  (SELECT COUNT(*) FROM healthcare_dev.silver_rejects.dim_beneficiarios r
    JOIN (SELECT last_load_id FROM v_quality_last_load_enriched WHERE table_name='dim_beneficiarios') x
      ON r.load_id = x.last_load_id
   GROUP BY reject_reason
   ORDER BY COUNT(*) DESC
   LIMIT 1);

##Relatório final (totais + último load + top reject do último load)

In [0]:
SELECT
  t.table_name,
  t.silver_count,
  t.rejects_count,
  t.rejects_pct,
  l.last_load_id,
  l.silver_count_last_load,
  l.rejects_count_last_load,
  l.rejects_pct_last_load,
  tr.top_reject_reason_last_load,
  tr.top_reject_count_last_load
FROM v_quality_totals_enriched t
LEFT JOIN v_quality_last_load_enriched l USING (table_name)
LEFT JOIN v_top_reject_last_load tr USING (table_name)
ORDER BY rejects_pct DESC, table_name;

##Checkpoints (última execução por tabela)

In [0]:
SELECT
  table_name,
  max(last_ingestion_ts) AS last_ingestion_ts,
  max(last_run_ts) AS last_run_ts,
  max(status) AS status
FROM healthcare_dev.silver_ops.pipeline_checkpoint
GROUP BY table_name
ORDER BY last_run_ts DESC;

##Persiste snapshot histórico (uma linha por tabela)

In [0]:
INSERT INTO healthcare_dev.silver_ops.quality_report_history
SELECT
  current_timestamp() AS report_ts,
  t.table_name,
  t.silver_count,
  t.rejects_count,
  t.rejects_pct,
  l.last_load_id,
  l.silver_count_last_load,
  l.rejects_count_last_load,
  l.rejects_pct_last_load,
  tr.top_reject_reason_last_load,
  tr.top_reject_count_last_load
FROM v_quality_totals_enriched t
LEFT JOIN v_quality_last_load_enriched l USING (table_name)
LEFT JOIN v_top_reject_last_load tr USING (table_name);